# Lab 6: nnU-Net v2 Self-Configuring Segmentation on SageMaker

This notebook demonstrates how to train a medical image segmentation model on Amazon SageMaker using nnU-Net v2. nnU-Net automatically determines optimal preprocessing, architecture, and training hyperparameters.

## What You'll Learn
- Running nnU-Net v2 on SageMaker with a custom Docker image
- Self-configuring preprocessing, training, and evaluation
- Comparing nnU-Net with manually configured MONAI models (Lab 1)

## Prerequisites
- Medical imaging data in S3 in nnU-Net format (`imagesTr/`, `labelsTr/`, `dataset.json`)
- SageMaker execution role with S3 access

## Step 1: Setup and Imports

In [ ]:
import os
import sagemaker
from sagemaker.estimator import Estimator
from sagemaker.local import LocalSession
import boto3

sagemaker_session = sagemaker.Session(boto3.Session(region_name='us-east-1'))
# Dedicated SageMaker execution role — use this when running outside a SageMaker notebook
region = sagemaker_session.boto_region_name
bucket = sagemaker_session.default_bucket()
print(f"SageMaker role: {role}")
print(f"Region: {region}")
print(f"Bucket: {bucket}")

## Step 1b: Create SageMaker Execution Role (one-time setup)

Run this cell once to create a dedicated SageMaker execution role with the same permissions as the default role. Skip if the role already exists.

In [ ]:
import sys
import os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('__file__')), '../..'))
from utils import get_or_create_role

role = get_or_create_role()
print(f"SageMaker role: {role}")


In [ ]:
# Remote S3 paths — used for the real SageMaker run in Step 5
data_bucket = "YOUR_BUCKET_NAME"  # Replace with your S3 bucket name
data_path = f"s3://{data_bucket}/nnUNet/"
output_path = f"s3://{bucket}/nnunet-segmentation/output"

# Local paths — used for local mode test in Step 4b
# Point local_data_path at a small dataset in nnU-Net format: imagesTr/, labelsTr/, dataset.json
local_data_path = os.path.abspath("../data/sample")
local_output_path = os.path.abspath("../output/local")
os.makedirs(local_output_path, exist_ok=True)

print(f"Remote training data : {data_path}")
print(f"Remote output path   : {output_path}")
print(f"Local training data  : {local_data_path}")
print(f"Local output path    : {local_output_path}")

## Step 3: Build and Push Docker Image

nnU-Net uses a separate Dockerfile since it has different dependencies than MONAI.

> **Local mode** only needs the local image tag — no ECR push required until Step 5.

In [ ]:
account_id = boto3.client('sts').get_caller_identity()['Account']
image_name = "nnunet-segmentation"
local_image = f"{image_name}:latest"
ecr_repo = f"{account_id}.dkr.ecr.{region}.amazonaws.com/{image_name}:latest"

print(f"Local image    : {local_image}")
print(f"ECR repository : {ecr_repo}")
print()
print("Build the image (required for both local mode and SageMaker):")
print(f"  cd ../code")
print(f"  docker build -f docker/Dockerfile.nnunet -t {local_image} .")
print()
print("Push to ECR (only needed for Step 5 — SageMaker remote run):")
print(f"  aws ecr get-login-password --region {region} | docker login --username AWS --password-stdin {account_id}.dkr.ecr.{region}.amazonaws.com")
print(f"  aws ecr create-repository --repository-name {image_name} --region {region} || true")
print(f"  docker tag {local_image} {ecr_repo}")
print(f"  docker push {ecr_repo}")

## Step 4a: Create SageMaker Estimator (remote)

This estimator targets a real SageMaker instance. It is used in Step 5.

In [ ]:
estimator = Estimator(
    image_uri=ecr_repo,
    entry_point="nnunet_pipeline.py",
    source_dir="../code/training/nnunet",
    role=role,
    instance_count=1,
    instance_type="ml.g5.xlarge",
    hyperparameters={
        "stages": "preprocess,train,evaluate",
        "num_epochs": 5
    },
    output_path=output_path,
    base_job_name="nnunet-pipeline",
    keep_alive_period_in_seconds=1800,
    sagemaker_session=sagemaker_session,
)
print("Remote estimator created successfully!")

## Step 4b: Local Mode Test

Runs the exact same container locally via Docker before submitting to SageMaker.
Validates the container entrypoint, script paths, and data loading without incurring cloud compute costs.

**Requirements:**
- Docker running locally
- `sagemaker[local]` installed: `pip install 'sagemaker[local]'`
- Image built locally (Step 3 build command — no push needed)
- Sample data at `../data/sample/` in nnU-Net format (`imagesTr/`, `labelsTr/`, `dataset.json`)

In [ ]:
# LocalSession must be used with instance_type="local" or "local_gpu"
local_session = LocalSession()
local_session.config = {
    'local': {
        'local_code': True,       # mount source_dir directly — no repackaging
        'container_config': {
            'shm_size': '8g',     # nnU-Net data workers need large shared memory
        }
    }
}

local_estimator = Estimator(
    image_uri=local_image,            # local Docker image — no ECR pull
    entry_point="nnunet_pipeline.py",
    source_dir="../code/training/nnunet",
    role=role,
    instance_count=1,
    instance_type="local_gpu",        # change to "local" if no GPU available
    hyperparameters={
        "stages": "preprocess,train,evaluate",
        "num_epochs": 5
    },
    output_path=f"file://{local_output_path}",
    sagemaker_session=local_session,
)
print("Local estimator created!")
print(f"  Image  : {local_image}")
print(f"  Data   : {local_data_path}")
print(f"  Output : {local_output_path}")

In [ ]:
# Kick off local training — uses file:// channel so no S3 access needed
local_estimator.fit(
    {"training": f"file://{local_data_path}"},
    wait=True,
    logs="All",
)

## Step 5: Start Training on SageMaker

Once local mode passes, submit the full job to SageMaker.

In [ ]:
estimator.fit({"training": data_path}, wait=True, logs="All")

## Step 6: View Training Results

In [ ]:
training_job_name = estimator.latest_training_job.name
model_data = estimator.model_data
print(f"Training job    : {training_job_name}")
print(f"Model artifacts : {model_data}")